## Construcción de la variable respuesta: clasificación de swings en lanzamientos de Baseball

### Exploración de la variable `description`

In [1]:
# Cargamos los paquetes necesarios 
import polars as pl
import pyprojroot

In [2]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")

El objetivo del trabajo consiste en predecir la probabilidad de que un bateador realice un swing ante un determinado lanzamiento. Para construir esta variable respuesta, se utiliza la variable `description`, la cual registra el resultado de cada pitcheo según Statcast. Debido a que esta variable no indica directamente si existió o no un intento de swing, resulta necesario establecer un criterio de clasificación que permita transformarla en una variable binaria. Para ello, se describen a continuación las categorías presentes en el conjunto de datos y su relación con la acción realizada por el bateador.

| Categoría | Descripción |
|-----------|-------------|
| `ball` | Lanzamiento fuera de la zona de strike que el bateador deja pasar sin intentar golpear. |
| `foul` | Contacto del bateador con la pelota que termina en territorio foul. |
| `hit_into_play` | Contacto del bateador que provoca que la pelota quede en juego. |
| `called_strike` | Lanzamiento determinado como strike por el árbitro sin que el bateador realice swing. |
| `swinging_strike` | Intento de swing en el que el bateador no logra hacer contacto con la pelota. |
| `blocked_ball` | Lanzamiento que es bloqueado por el receptor sin intervención del bateador. |
| `foul_tip` | Contacto leve del bateador con la pelota que termina siendo atrapado por el receptor. |
| `swinging_strike_blocked` | Intento de swing fallido donde la pelota es bloqueada por el receptor. |
| `hit_by_pitch` | El lanzamiento golpea al bateador sin que este intente golpear la pelota. |
| `foul_bunt` | Intento de bunt donde la pelota termina en territorio foul. |
| `missed_bunt` | Intento de bunt en el que el bateador no logra hacer contacto con la pelota. |
| `bunt_foul_tip` | Intento de bunt que genera un contacto mínimo y termina como foul tip. |
| `pitchout` | Lanzamiento intencional fuera de la zona de strike, generalmente utilizado para evitar una situación ofensiva específica. |
| `foul_pitchout` | Lanzamiento intencional (*pitchout*) que termina en foul debido a la acción del bateador. |

Luego de visualizar las distintas categorías de la variable, se construye una lista de tuplas que permite visualizar el nombre de la categoría y la frecuencia de cada una de ellas.

In [3]:
datos_entrenamiento["description"].value_counts()\
    .sort("count", descending=True)\
    .rows()

[('ball', 236921),
 ('foul', 125997),
 ('hit_into_play', 121707),
 ('called_strike', 116618),
 ('swinging_strike', 75731),
 ('blocked_ball', 17316),
 ('foul_tip', 6732),
 ('swinging_strike_blocked', 4713),
 ('hit_by_pitch', 2112),
 ('foul_bunt', 1569),
 ('missed_bunt', 361),
 ('bunt_foul_tip', 41),
 ('pitchout', 33),
 ('foul_pitchout', 1)]

### Construcción de la variable respuesta


Para definir el concepto de *swing* se utiliza la definición propuesta por Baseball Almanac, donde se describe como *"the action of a batter in attempting to hit the ball"*, es decir, la acción del bateador al intentar golpear la pelota. Esta definición resulta adecuada para el problema planteado, ya que permite considerar como swing tanto los lanzamientos donde existe contacto con la pelota como aquellos donde el bateador intenta golpearla pero no logra establecer contacto.

A partir de esta definición, los eventos registrados como `hit_into_play` se consideran intentos de swing, ya que representan situaciones donde el bateador logra contactar la pelota y esta continúa en juego. De la misma manera, las categorías `foul` y `foul_tip` evidencian una acción del bateador sobre el lanzamiento, debido a que el contacto con la pelota ocurre durante un intento de bateo.

Además, los eventos `swinging_strike` y `swinging_strike_blocked` representan situaciones donde el bateador realiza un intento de swing, pero no logra establecer contacto con la pelota. La diferencia entre ambas categorías es que, en el segundo caso, el lanzamiento es posteriormente bloqueado por el receptor. Debido a que ambas situaciones implican una acción activa del bateador sobre el lanzamiento, se clasifican como intentos de swing.

Los eventos asociados al bunt requieren una consideración particular. Según Baseball Almanac, un *bunt* consiste en una acción donde el bateador busca contactar intencionalmente la pelota mediante un control del bate, sin realizar un swing convencional. Aunque esta acción difiere del movimiento tradicional del swing, representa una decisión voluntaria del bateador de intervenir sobre el lanzamiento. Por este motivo, las categorías `foul_bunt`, `missed_bunt` y `bunt_foul_tip` son consideradas como intentos de swing para la construcción de la variable respuesta.

Por otra parte, existen eventos donde no se observa una acción voluntaria del bateador sobre el lanzamiento. En el caso de `ball`, el bateador deja pasar la pelota sin intentar golpearla, por lo que no se considera un swing. La **Regla 5.05(b)(1)** del reglamento oficial de MLB establece que, luego de recibir cuatro lanzamientos fuera de la zona de strike (*base on balls*), el bateador obtiene la primera base.

De manera similar, los eventos `hit_by_pitch` corresponden a situaciones donde la pelota golpea al bateador sin que este intente realizar un swing. La **Regla 5.05(b)(2)** establece que el bateador recibe la primera base cuando es golpeado por un lanzamiento que no estaba intentando batear. Por este motivo, ambas categorías se clasifican como ausencia de swing.

También resulta necesario considerar la definición de *strike*, ya que no todos los strikes implican una acción del bateador. Según Baseball Almanac, un *strike* corresponde a un lanzamiento válido que es considerado como tal por el árbitro debido a alguna de las siguientes situaciones: el bateador realiza un swing y falla, el lanzamiento atraviesa la zona de strike sin que el bateador intente golpearlo, la pelota es bateada hacia territorio foul bajo determinadas condiciones, se realiza un bunt foul con dos strikes o se produce un *foul tip* atrapado por el receptor con dos strikes.

Esta definición permite distinguir entre aquellos strikes generados por una acción del bateador y aquellos donde el bateador decide no intervenir sobre el lanzamiento. Por este motivo, los eventos `called_strike` se clasifican como ausencia de swing, ya que corresponden a lanzamientos donde el bateador no intenta golpear la pelota y el árbitro determina que el lanzamiento fue strike.

Además, la categoría `blocked_ball` corresponde a lanzamientos que son bloqueados por el receptor sin que exista evidencia de un intento de swing por parte del bateador. Por otra parte, `pitchout` representa un lanzamiento intencionalmente alejado de la zona de bateo, utilizado generalmente por la defensa para evitar una situación ofensiva específica, y en el cual el bateador no realiza una acción sobre la pelota.

Finalmente, la categoría `foul_pitchout` se considera un intento de swing, ya que, aunque el lanzamiento corresponde a un *pitchout*, la presencia de una pelota foul indica que el bateador realizó una acción sobre el lanzamiento.


### Creación de la variable respuesta

A partir lo explicado anteriormente se define la variable respuesta:

| Valor | Categorías de `description` |
| ----------- | ------------------------------------------------------------------------------------------------------------------------------ |
| `swing = 1` | `foul`, `hit_into_play`, `swinging_strike`, `swinging_strike_blocked`, `foul_tip`, `foul_bunt`, `missed_bunt`, `bunt_foul_tip`, `foul_pitchout` |
| `swing = 0` | `ball`, `called_strike`, `blocked_ball`, `hit_by_pitch`, `pitchout` |

In [4]:
categorias_swing = [
    "foul",
    "hit_into_play",
    "swinging_strike",
    "swinging_strike_blocked",
    "foul_tip",
    "foul_bunt",
    "missed_bunt",
    "bunt_foul_tip",
    "foul_pitchout"
]

datos_entrenamiento = datos_entrenamiento.with_columns(
    pl.when(pl.col("description").is_in(categorias_swing))
    .then(1)
    .otherwise(0)
    .alias("swing")
)

datos_entrenamiento["swing"].value_counts()

swing,count
i32,u32
0,373000
1,336852


Una vez definida y construida la variable respuesta, se guarda el dataframe para seguir con el análisis en las demas notebooks.

In [5]:
datos_entrenamiento.write_parquet(
    "../datos/temporada1.parquet"
)

### Bibliografía



* Baseball Almanac. *Baseball Dictionary: Swing*. Disponible en:
  https://www.baseball-almanac.com/dictionary-term.php?term=swing

* Baseball Almanac. *Baseball Dictionary: Bunt*. Disponible en:
  https://www.baseball-almanac.com/dictionary-term.php?term=bunt

* Baseball Almanac. *Baseball Dictionary: Strike*. Disponible en:
  https://www.baseball-almanac.com/dictionary-term.php?term=strike

* Major League Baseball. *Official Baseball Rules 2025*. Disponible en:
  [[Link al pdf](https://mktg.mlbstatic.com/mlb/official-information/2025-official-baseball-rules.pdf)]
